# log 文件可视化（训练曲线对比）

**回答的问题（对应预训练边际效益讨论的零成本分析）：**
1. **Run A vs Run B 收益判定**：val mIoU 曲线叠加，预训练初始化是否翻正/缩小差距；
2. **欠训练 vs 干扰判别**：末端斜率——DiffAE 臂仍在爬升而监督臂已平 → 欠训练型（值得续训）；两条都平但仍有差距 → 干扰/结构型（加预算救不了）；
3. **扩散侧饱和程度**：`loss_diffae` 是否还在降、`zsem_std` 是否回升；
4. **收敛速度对比**（tsample 的"加速收敛"视角）：time-to-threshold（达到最终 mIoU 90% 所需 iters）与 AUC。

**日志来源**：mmengine 的 `work_dirs/<exp>/<时间戳>/vis_data/*.json`（每行一条 JSON 标量记录）。
找不到 json 时自动降级解析同目录 `.log` 文本。断点续训产生的多个时间戳目录会自动按修改时间合并。

**依赖**：只需 `numpy + matplotlib`，不需要 mmseg 环境，任意 python3 内核都能跑。

In [ ]:
# ============================================================
# 配置区：把要对比的 run 写在这里（label -> 路径）
# 路径支持三种写法：
#   1. work_dir 根目录（自动递归找 vis_data/*.json，找不到再找 *.log）
#   2. 具体的 vis_data/*.json 文件
#   3. 具体的 *.log 文本文件
# ============================================================
import os
import re
import json
import glob
import numpy as np
import matplotlib.pyplot as plt

RUNS = {
    # 'baseline_80k':  '../tasks/Segmentation/mmsegmentation/work_dirs/eden/fcn_baseline',
    # 'runA_diffae':   '../tasks/Segmentation/mmsegmentation/work_dirs/eden/fcn_diffae',
    # 'runB_pre_ft':   '../tasks/Segmentation/mmsegmentation/work_dirs/eden/fcn_diffae_pretrain_ft',
    # 'pretrain_only': '../tasks/Segmentation/mmsegmentation/work_dirs/eden/fcn_diffae_pretrain',
}

SMOOTH = 0.6     # 训练 loss 的 EMA 平滑系数，0 = 不平滑
TAIL_N = 3       # 末端斜率用最后几个 val 点线性拟合
THRESH = 0.9     # time-to-threshold：达到 best mIoU 的 90% 所需的 iter

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'WenQuanYi Zen Hei',
                                   'SimHei', 'Microsoft YaHei', 'DejaVu Sans']

In [ ]:
# ------------------------------------------------------------
# 解析器：mmengine 标量日志 -> {'train': {key: (steps, vals)}, 'val': {...}}
# ------------------------------------------------------------
_KV = re.compile(r'([A-Za-z0-9_.]+): (-?[\d.]+(?:[eE][-+]?\d+)?)')
_TRAIN_ITER = re.compile(r'Iter\(train\)\s*\[\s*(\d+)/')


def _iter_records(path):
    """逐行产出标量 dict。json 按行解析；.log 文本用正则提取 key: value。"""
    last_train_step = None
    with open(path, encoding='utf-8', errors='ignore') as fh:
        if path.endswith('.json'):
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if isinstance(rec, dict):
                    yield rec, None
        else:
            for line in fh:
                m = _TRAIN_ITER.search(line)
                if m:
                    last_train_step = int(m.group(1))
                if 'Iter(' not in line:
                    continue
                rec = {k: float(v) for k, v in _KV.findall(line)}
                yield rec, last_train_step


def load_run(path):
    """读一个 run 的全部标量（自动合并断点续训的多个时间戳目录）。"""
    if os.path.isfile(path):
        files = [path]
    else:
        files = sorted(glob.glob(os.path.join(path, '**', 'vis_data', '*.json'),
                                 recursive=True), key=os.path.getmtime)
        if not files:
            files = sorted(glob.glob(os.path.join(path, '**', '*.log'),
                                     recursive=True), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(f'{path} 下没有找到 vis_data/*.json 或 *.log')

    train, val = {}, {}
    for f in files:
        for rec, last_step in _iter_records(f):
            rec = dict(rec)
            step = rec.pop('step', None)
            if step is None:
                step = last_step          # .log 的 val 行没有 step，用最近的 train iter
            if step is None:
                continue
            dst = val if ('mIoU' in rec or 'aAcc' in rec) else train
            for k, v in rec.items():
                if isinstance(v, (int, float)) and np.isfinite(v):
                    dst.setdefault(k, {})[int(step)] = float(v)   # 同 step 后写覆盖先写

    out = {}
    for name, d in (('train', train), ('val', val)):
        out[name] = {}
        for k, pairs in d.items():
            ss = sorted(pairs)
            out[name][k] = (np.array(ss), np.array([pairs[s] for s in ss]))
    return out


def load_runs(runs_cfg):
    data = {}
    for label, path in runs_cfg.items():
        data[label] = load_run(path)
        tr, va = data[label]['train'], data[label]['val']
        print(f'[{label}] train keys: {sorted(tr)}')
        print(f'{" " * len(label)}  val   keys: {sorted(va)}  ({len(va.get("mIoU", ([], []))[0])} 个 val 点)')
    return data


if not RUNS:
    print('请先在配置区填写 RUNS（label -> work_dir 路径）')
else:
    DATA = load_runs(RUNS)

In [ ]:
# ------------------------------------------------------------
# 图 1：验证集指标对比（Run A vs Run B 判定主图）
# ------------------------------------------------------------
def plot_val(data, metrics=('mIoU', 'mAcc', 'aAcc')):
    metrics = [m for m in metrics if any(m in d['val'] for d in data.values())]
    if not metrics:
        print('没有 val 指标可画（pretrain 臂不做验证是正常的）')
        return
    fig, axes = plt.subplots(1, len(metrics), figsize=(5.2 * len(metrics), 4.2))
    for ax, m in zip(np.atleast_1d(axes), metrics):
        for label, d in data.items():
            if m not in d['val']:
                continue
            s, v = d['val'][m]
            tag = f'{label}  best {v.max():.2f} @ {int(s[v.argmax()]) // 1000}k'
            ax.plot(s, v, marker='o', ms=4, lw=1.4, label=tag)
        ax.set_title(m)
        ax.set_xlabel('iter')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle('val metrics', y=1.02)
    fig.tight_layout()
    plt.show()


if RUNS:
    plot_val(DATA)

In [ ]:
# ------------------------------------------------------------
# 图 2：训练 loss（EMA 平滑；淡色为原始值）
# 判据：loss_diffae 到 80k 是否还在明显下降（还在降 -> 欠训练）
# ------------------------------------------------------------
def ema(v, w):
    if not w or len(v) == 0:
        return v
    out = np.empty_like(v, dtype=float)
    out[0] = v[0]
    for i in range(1, len(v)):
        out[i] = w * out[i - 1] + (1 - w) * v[i]
    return out


DEFAULT_LOSS_KEYS = ['loss', 'decode.loss_ce', 'aux.loss_ce', 'loss_diffae']


def plot_train(data, keys=None, smooth=SMOOTH):
    keys = keys or [k for k in DEFAULT_LOSS_KEYS
                    if any(k in d['train'] for d in data.values())]
    if not keys:
        print('没有 loss 标量可画')
        return
    fig, axes = plt.subplots(1, len(keys), figsize=(5.2 * len(keys), 4.2))
    for ax, k in zip(np.atleast_1d(axes), keys):
        for label, d in data.items():
            if k not in d['train']:
                continue
            s, v = d['train'][k]
            (ln,) = ax.plot(s, ema(v, smooth), lw=1.3, label=label)
            if smooth:
                ax.plot(s, v, lw=0.4, alpha=0.15, color=ln.get_color())
        ax.set_title(k)
        ax.set_xlabel('iter')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(f'training losses (EMA {smooth})', y=1.02)
    fig.tight_layout()
    plt.show()


if RUNS:
    plot_train(DATA)

In [ ]:
# ------------------------------------------------------------
# 图 3：监控标量（zsem_std 对数轴）
# 判据：zsem_std 从 1e-5 量级回升 -> z_sem 存活，扩散分支真正在驱动 encoder
# ------------------------------------------------------------
MONITOR_KEYS = [('zsem_std', True), ('tw_ent', False), ('tw_max', False),
                ('ts_mean', False), ('lr', False)]


def plot_monitors(data, keys=MONITOR_KEYS):
    keys = [(k, lg) for k, lg in keys if any(k in d['train'] for d in data.values())]
    if not keys:
        print('没有监控标量可画')
        return
    fig, axes = plt.subplots(1, len(keys), figsize=(4.6 * len(keys), 4))
    for ax, (k, lg) in zip(np.atleast_1d(axes), keys):
        for label, d in data.items():
            if k not in d['train']:
                continue
            s, v = d['train'][k]
            ax.plot(s, v, lw=1.0, label=label)
        if lg:
            ax.set_yscale('log')
        ax.set_title(k + (' (log)' if lg else ''))
        ax.set_xlabel('iter')
        ax.grid(alpha=0.3, which='both' if lg else 'major')
        ax.legend(fontsize=8)
    fig.suptitle('monitor scalars', y=1.02)
    fig.tight_layout()
    plt.show()


if RUNS:
    plot_monitors(DATA)

In [ ]:
# ------------------------------------------------------------
# 汇总表：best/final mIoU、末端斜率、time-to-threshold、AUC
#
# 解读（对应欠训练 vs 干扰判别）：
#   tail_slope 明显 > 0 且 DiffAE 臂 > baseline  -> 欠训练型：值得单点续训穿越实验
#   两臂 tail_slope 都 ≈ 0 但仍有差距           -> 干扰/结构型：矛头指向 lr 配比/lambda
#   TTT 更小 / AUC 更大                          -> 收敛更快（tsample 加速视角的度量）
# ------------------------------------------------------------
_trapezoid = getattr(np, 'trapezoid', None) or np.trapz


def summary(data, tail_n=TAIL_N, thresh=THRESH):
    rows = []
    for label, d in data.items():
        if 'mIoU' not in d['val'] or len(d['val']['mIoU'][0]) == 0:
            rows.append((label, *[float('nan')] * 6, None))
            continue
        s, v = d['val']['mIoU']
        best, best_at = float(v.max()), int(s[v.argmax()])
        final = float(v[-1])
        k = min(tail_n, len(s))
        slope = float(np.polyfit(s[-k:], v[-k:], 1)[0] * 1e4) if k >= 2 else float('nan')
        hit = v >= thresh * best
        ttt = int(s[np.argmax(hit)]) if hit.any() else None
        auc = float(_trapezoid(v, s) / (s[-1] - s[0])) if len(s) > 1 else float('nan')
        rows.append((label, best, best_at, final, slope, ttt, auc))

    hdr = f'{"run":<16}{"best":>7}{"@iter":>8}{"final":>8}{"slope/10k":>11}{"TTT90%":>9}{"AUC":>8}'
    print(hdr)
    print('-' * len(hdr))
    for r in rows:
        label, best, best_at, final, slope, ttt, auc = r
        ttt_s = f'{ttt // 1000}k' if ttt is not None else '-'
        at_s = f'{best_at // 1000}k' if best_at else '-'
        print(f'{label:<16}{best:>7.2f}{at_s:>8}{final:>8.2f}{slope:>11.3f}{ttt_s:>9}{auc:>8.2f}')

    print()
    print(f'slope/10k：最后 {tail_n} 个 val 点线性拟合斜率（mIoU / 10k iters），正值=还在涨')
    print(f'TTT90%：首次达到 best 的 {thresh * 100:.0f}% 的 iter；AUC：val 曲线下的平均 mIoU')


if RUNS:
    summary(DATA)